# Создание `df_ind` и агрегированных комментариев

Ноутбук предполагает, что `df_out` уже создан и факторный анализ рассчитан.

### Фильтр комментариев

В комментарий включаются только те договоры, у которых общий эффект `Ухудшение качества + Переоценка + Изменение портфеля` по модулю **строго больше 0,1 млн BYN**. Отдельные составляющие эффекта внутри такого договора показываются без порога 0,1.

In [ ]:

import re
import numpy as np
import pandas as pd

COL_SEGMENT = "СБЛ"
COL_UNP = "УНП"
COL_CLIENT = "Наименование клиента"
COL_CONTRACT = "Номер договора"
COL_CURRENCY = "Валюта"

MONTH_THRESHOLD = 0.1
EPS = 1e-7
COMMENT_EFFECT_THRESHOLD = 0.1
EFFECT_EPS = 1e-6

CLIENT_NAME_OVERRIDE = {
    # "ВСТАВЬ_СЮДА_УНП": "ВСТАВЬ_СЮДА_НАЗВАНИЕ_КЛИЕНТА",
}

if "df_out" not in globals():
    raise NameError("Сначала должен быть создан dataframe df_out")

if "report_dates" not in globals():
    report_dates = []
    for col in df_out.columns:
        match = re.match(r"^Задолженность_(\d{2}\.\d{2}\.\d{4})$", str(col))
        if match:
            report_dates.append(match.group(1))
    report_dates = sorted(
        set(report_dates),
        key=lambda x: pd.to_datetime(x, format="%d.%m.%Y"),
    )

if not report_dates:
    raise ValueError("Не найдены отчетные даты")


def first_not_null(series):
    values = series.dropna()
    return values.iloc[0] if len(values) else np.nan


def normalize_unp(value):
    if pd.isna(value):
        return ""
    value = str(value).strip()
    if value.endswith(".0"):
        try:
            value = str(int(float(value)))
        except Exception:
            pass
    return value


def to_number(value):
    if pd.isna(value):
        return 0.0
    try:
        return float(
            str(value).strip().replace("\xa0", "").replace(" ", "").replace(",", ".")
        )
    except Exception:
        return 0.0


def clean_text(value):
    if pd.isna(value):
        return ""
    value = str(value).strip()
    return "" if value.lower() in {"", "nan", "none"} else value


def normalized_text(value):
    return clean_text(value).lower().replace("ё", "е")


def is_active(value):
    if pd.isna(value):
        return False
    try:
        return float(value) == 1
    except Exception:
        return normalized_text(value) in {"да", "есть", "true", "yes", "+"}


def has_restra(value):
    if is_active(value):
        return True
    text = normalized_text(value)
    return "рестр" in text or "реестр" in text


def display_value(value):
    value = clean_text(value)
    return value if value else "нет"


def fmt_mln_raw(value):
    return f"{to_number(value) / 1_000_000:.2f}".replace(".", ",")


def fmt_effect(value):
    return f"{value:+.2f}".replace(".", ",")


def fmt_rate(value):
    number = to_number(value)
    if abs(number - round(number)) < EPS:
        return str(int(round(number)))
    return f"{number:.4f}".rstrip("0").rstrip(".").replace(".", ",")


# ============================================================
# СОЗДАНИЕ df_ind
# ============================================================

df_ind = (
    df_out
    .groupby(COL_UNP, dropna=False, as_index=False)
    .agg({
        COL_SEGMENT: first_not_null,
        COL_CLIENT: first_not_null,
    })
)

df_ind[COL_CLIENT] = [
    CLIENT_NAME_OVERRIDE.get(normalize_unp(unp), client_name)
    for unp, client_name in zip(df_ind[COL_UNP], df_ind[COL_CLIENT])
]

sum_columns = []

for date in report_dates:
    quality_col = f"Ухудшение качества_{date}"
    revaluation_col = f"Переоценка_{date}"
    portfolio_col = f"Изменение портфеля_{date}"
    result_col = f"Сумма_{date}"

    for col in [quality_col, revaluation_col, portfolio_col]:
        if col not in df_out.columns:
            raise KeyError(f"Отсутствует столбец: {col}")

    row_sum = (
        pd.to_numeric(df_out[quality_col], errors="coerce").fillna(0.0)
        + pd.to_numeric(df_out[revaluation_col], errors="coerce").fillna(0.0)
        + pd.to_numeric(df_out[portfolio_col], errors="coerce").fillna(0.0)
    ) / 1_000_000

    temp = pd.DataFrame({
        COL_UNP: df_out[COL_UNP],
        result_col: row_sum,
    })

    temp = (
        temp
        .groupby(COL_UNP, dropna=False, as_index=False)[result_col]
        .sum()
    )

    df_ind = df_ind.merge(temp, on=COL_UNP, how="left")
    sum_columns.append(result_col)

df_ind[sum_columns] = df_ind[sum_columns].fillna(0.0)

df_ind["Инд"] = (
    df_ind[sum_columns]
    .abs()
    .gt(MONTH_THRESHOLD)
    .any(axis=1)
    .astype(int)
)


# ============================================================
# КОММЕНТАРИИ
# ============================================================

def get_factor_changes(row, prev_date, current_date):
    changes = []

    for factor, came, left in [
        ("НИ", "пришла НИ", "ушла НИ"),
        ("ПФН", "пришел ПФН", "ушел ПФН"),
        ("НВВ", "пришло НВВ", "ушло НВВ"),
    ]:
        prev_v = is_active(row[f"{factor}_{prev_date}"])
        curr_v = is_active(row[f"{factor}_{current_date}"])

        if not prev_v and curr_v:
            changes.append(came)
        elif prev_v and not curr_v:
            changes.append(left)

    prev_restra = row[f"Рестра_{prev_date}"]
    curr_restra = row[f"Рестра_{current_date}"]

    if not has_restra(prev_restra) and has_restra(curr_restra):
        changes.append("появилась рестра")
    elif has_restra(prev_restra) and not has_restra(curr_restra):
        changes.append("ушла рестра")
    elif normalized_text(prev_restra) != normalized_text(curr_restra):
        changes.append(
            f"рестра: {display_value(prev_restra)} → {display_value(curr_restra)}"
        )

    prev_sec = row[f"Обеспеченность_{prev_date}"]
    curr_sec = row[f"Обеспеченность_{current_date}"]

    if normalized_text(prev_sec) != normalized_text(curr_sec):
        changes.append(
            f"обеспеченность: {display_value(prev_sec)} → {display_value(curr_sec)}"
        )

    prev_gr = row[f"ГР_{prev_date}"]
    curr_gr = row[f"ГР_{current_date}"]

    if normalized_text(prev_gr) != normalized_text(curr_gr):
        changes.append(
            f"ГР: {display_value(prev_gr)} → {display_value(curr_gr)}"
        )

    prev_rate = to_number(row[f"%рез_{prev_date}"])
    curr_rate = to_number(row[f"%рез_{current_date}"])

    if abs(curr_rate - prev_rate) > EPS:
        changes.append(
            f"%рез: {fmt_rate(prev_rate)}% → {fmt_rate(curr_rate)}%"
        )

    return changes


def current_characteristics(row, current_date):
    result = []

    if is_active(row[f"НИ_{current_date}"]):
        result.append("НИ")
    if is_active(row[f"ПФН_{current_date}"]):
        result.append("ПФН")
    if is_active(row[f"НВВ_{current_date}"]):
        result.append("НВВ")
    if has_restra(row[f"Рестра_{current_date}"]):
        result.append("рестра")

    security = clean_text(row[f"Обеспеченность_{current_date}"])
    if security:
        result.append(f"обеспеченность: {security}")

    return result


def build_contract_comment(row, prev_date, current_date):
    contract = clean_text(row[COL_CONTRACT])
    currency = clean_text(row[COL_CURRENCY])

    prev_debt = to_number(row[f"Задолженность_{prev_date}"])
    curr_debt = to_number(row[f"Задолженность_{current_date}"])

    quality = to_number(row[f"Ухудшение качества_{current_date}"]) / 1_000_000
    revaluation = to_number(row[f"Переоценка_{current_date}"]) / 1_000_000
    portfolio = to_number(row[f"Изменение портфеля_{current_date}"]) / 1_000_000
    effect = quality + revaluation + portfolio

    if abs(effect) <= COMMENT_EFFECT_THRESHOLD:
        return ""

    contract_text = f"№{contract}" if contract else "без номера"

    if abs(prev_debt) <= EPS and abs(curr_debt) > EPS:
        gr = display_value(row[f"ГР_{current_date}"])
        characteristics = current_characteristics(row, current_date)
        char_text = f" ({', '.join(characteristics)})" if characteristics else ""
        reason = (
            f"новый договор {contract_text}: "
            f"{fmt_mln_raw(curr_debt)} млн {currency}, ГР {gr}{char_text}"
        )

    elif abs(prev_debt) > EPS and abs(curr_debt) <= EPS:
        reason = (
            f"полное погашение договора {contract_text}: "
            f"{fmt_mln_raw(prev_debt)} млн {currency} → 0"
        )

    else:
        parts = []
        delta = curr_debt - prev_debt

        if delta < -EPS:
            parts.append(
                f"погашение по договору {contract_text}: "
                f"{fmt_mln_raw(prev_debt)} → {fmt_mln_raw(curr_debt)} млн {currency}"
            )
        elif delta > EPS:
            parts.append(
                f"увеличение остатка по договору {contract_text}: "
                f"{fmt_mln_raw(prev_debt)} → {fmt_mln_raw(curr_debt)} млн {currency}"
            )

        parts.extend(get_factor_changes(row, prev_date, current_date))

        if not parts and abs(revaluation) > EFFECT_EPS:
            parts.append(f"валютная переоценка по договору {contract_text}")

        if not parts:
            parts.append(f"изменение расходов по договору {contract_text}")

        reason = "; ".join(parts)

    effect_parts = []

    if abs(quality) > EFFECT_EPS:
        effect_parts.append(f"качество {fmt_effect(quality)}")
    if abs(revaluation) > EFFECT_EPS:
        effect_parts.append(f"переоценка {fmt_effect(revaluation)}")
    if abs(portfolio) > EFFECT_EPS:
        effect_parts.append(f"портфель {fmt_effect(portfolio)}")

    detail = f" ({', '.join(effect_parts)})" if effect_parts else ""

    return f"{reason}; эффект {fmt_effect(effect)} млн BYN{detail}"


def build_unp_comment(row_ind):
    if to_number(row_ind["Инд"]) != 1:
        return ""

    unp = normalize_unp(row_ind[COL_UNP])

    rows = df_out[
        df_out[COL_UNP].map(normalize_unp).eq(unp)
    ]

    month_comments = []

    for prev_date, current_date in zip(report_dates[:-1], report_dates[1:]):
        sum_col = f"Сумма_{current_date}"

        if sum_col not in df_ind.columns:
            continue

        month_effect = to_number(row_ind[sum_col])

        if abs(month_effect) <= MONTH_THRESHOLD:
            continue

        contract_comments = []

        for _, contract_row in rows.iterrows():
            comment = build_contract_comment(
                contract_row,
                prev_date,
                current_date,
            )

            if comment:
                contract_comments.append(comment)

        if contract_comments:
            month_text = (
                f"{current_date}: итого {fmt_effect(month_effect)} млн BYN — "
                + " | ".join(contract_comments)
            )
        else:
            month_text = (
                f"{current_date}: итого {fmt_effect(month_effect)} млн BYN"
            )

        month_comments.append(month_text)

    return "\n".join(month_comments)


df_ind["Комментарий"] = df_ind.apply(
    build_unp_comment,
    axis=1,
)

df_ind = df_ind[
    [
        COL_SEGMENT,
        COL_UNP,
        COL_CLIENT,
        *sum_columns,
        "Инд",
        "Комментарий",
    ]
]

display(df_ind)
